# 01b. QuartzNet + WordAux — вспомогательный CTC-loss на уровне слов

## Идея

Помимо основного char-CTC, добавляем вспомогательный CTC-loss, который работает
на уровне слов (WordAuxCTCHead). Энкодер проецирует свои финальные фичи в
пространство closed word-vocabulary (43 слова + blank).

Это даёт дополнительный градиентный сигнал: модель обучается одновременно
предсказывать символы и слова — эффект схож с multi-task learning.

**Адаптация под Wave-1 Batch:** поля `word_targets` / `word_target_lengths` удалены из `Batch`.
Word-targets вычисляются **на лету в цикле** из `batch.transcriptions` через `digits_to_words` + `WordVocab.encode`.
`WordVocab` и `NUMBER_WORDS` — inline в этом ноутбуке (удалены из src/ в Wave 1).

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Локально: если репо уже установлен (uv sync / pip install -e .), эта ячейка — no-op.

# --- Colab ---
# !git clone --depth 1 https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git
# import sys
# sys.path.insert(0, "ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

# --- Kaggle ---
# import sys
# sys.path.insert(0, "/kaggle/working/ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

print("Deps cell — fill in your platform block above if on Colab/Kaggle.")

## Пути

`DATA_ROOT`, `CKPT_ROOT` и сопутствующие пути автоматически резолвятся в ячейке ниже относительно расположения ноутбука (`notebooks/experiments/..`). Правки требуются, только если структура репозитория изменена.

In [ ]:
# Idempotent data download: populates ../data/ with train/ dev/ test/ and
# their CSVs from the shared Google Drive ZIP. No-op if already present.
from pathlib import Path
import zipfile

import gdown

NOTEBOOK_DIR = Path.cwd()  # notebooks/experiments/
DATA_ROOT = (NOTEBOOK_DIR / ".." / "data").resolve()
ZIP_PATH = (NOTEBOOK_DIR / ".." / "data.zip").resolve()

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        gdown.download(
            url="https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link",
            output=str(ZIP_PATH),
            quiet=False,
            fuzzy=True,
        )
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT.parent)
    print(f"Extracted to {DATA_ROOT}")
else:
    print(f"Data already present at {DATA_ROOT}")


In [ ]:
# Env hardening — MUST run before `import torch` in this process.
# PYTORCH_CUDA_ALLOC_CONF reduces VRAM fragmentation on torch.compile +
# variable T batches; cudnn.benchmark=False avoids autotune thrash under
# padded, variable-length inputs; matmul_precision="high" enables TF32.
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("high")

# Limit torch.compile graph cache (dynamic=True can cache 50+ variants).
import torch._dynamo
torch._dynamo.config.cache_size_limit = 8


In [ ]:
# Paths — auto-resolved from DATA_ROOT defined in the download cell.
from pathlib import Path
import torch

TRAIN_ROOT = DATA_ROOT / "train"
DEV_ROOT = DATA_ROOT / "dev"
_TEST_DIR = DATA_ROOT / "test"
TEST_ROOT: Path | None = _TEST_DIR if _TEST_DIR.exists() else None
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = (Path.cwd() / ".." / ".." / "checkpoints" / "01b_word_aux").resolve()

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

MUSAN_ROOT: Path | None = Path.home() / "datasets" / "musan" / "noise"
RIR_ROOT: Path | None = Path.home() / "datasets" / "RIRS_NOISES" / "simulated_rirs"
if MUSAN_ROOT is not None and not MUSAN_ROOT.exists():
    print(f"[warn] MUSAN not found at {MUSAN_ROOT} — AddNoise disabled")
    MUSAN_ROOT = None
if RIR_ROOT is not None and not RIR_ROOT.exists():
    print(f"[warn] RIR not found at {RIR_ROOT} — RIR disabled")
    RIR_ROOT = None

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from gp1.data.manifest import records_from_csv
from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.audio_aug_gpu import GPUAudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import save_best
from gp1.train.metrics import compute_cer, compute_cer_in_out_harmonic
from gp1.decoding.greedy import greedy_decode
from gp1.text.vocab import CharVocab
from gp1.text.normalize import digits_to_words
from gp1.types import AugConfig

In [ ]:
# ---------------------------------------------------------------------------
# WordVocab + NUMBER_WORDS — скопировано из src/gp1/text/vocab_word.py (удалён в Wave 1)
# ---------------------------------------------------------------------------

# Exhaustive set of Russian number words for range 1000..999999.
NUMBER_WORDS: tuple[str, ...] = (
    "ноль",
    "один",
    "одна",
    "два",
    "две",
    "три",
    "четыре",
    "пять",
    "шесть",
    "семь",
    "восемь",
    "девять",
    "десять",
    "одиннадцать",
    "двенадцать",
    "тринадцать",
    "четырнадцать",
    "пятнадцать",
    "шестнадцать",
    "семнадцать",
    "восемнадцать",
    "девятнадцать",
    "двадцать",
    "тридцать",
    "сорок",
    "пятьдесят",
    "шестьдесят",
    "семьдесят",
    "восемьдесят",
    "девяносто",
    "сто",
    "двести",
    "триста",
    "четыреста",
    "пятьсот",
    "шестьсот",
    "семьсот",
    "восемьсот",
    "девятьсот",
    "тысяча",
    "тысячи",
    "тысяч",
)

_WORD_TO_ID: dict[str, int] = {word: idx + 1 for idx, word in enumerate(NUMBER_WORDS)}
_ID_TO_WORD: dict[int, str] = {v: k for k, v in _WORD_TO_ID.items()}


class WordVocab:
    """Closed word vocabulary for Russian number words. BLANK_ID = 0."""

    BLANK_ID: int = 0

    @property
    def size(self) -> int:
        return 1 + len(NUMBER_WORDS)

    def encode(self, text: str) -> list[int]:
        """Encode space-separated Russian number words to token ids."""
        if not text:
            return []
        result: list[int] = []
        for word in text.split():
            if word not in _WORD_TO_ID:
                raise ValueError(f"Word {word!r} is not in the closed vocabulary.")
            result.append(_WORD_TO_ID[word])
        return result

    def decode(self, ids: list[int]) -> str:
        """Decode token ids to space-separated word string (strips blank ids)."""
        if not ids:
            return ""
        words: list[str] = []
        for token_id in ids:
            if token_id == self.BLANK_ID:
                continue
            words.append(_ID_TO_WORD[token_id])
        return " ".join(words)

In [ ]:
# ---------------------------------------------------------------------------
# WordAuxCTCHead — скопировано из src/gp1/losses/word_aux.py (удалён в Wave 1)
# Адаптация: использует out.intermediate вместо отдельного поля Batch.
# ---------------------------------------------------------------------------

class WordAuxCTCHead(nn.Module):
    """Auxiliary CTC head for word-level targets applied to encoder features.

    Adapted for Wave-1 Batch: word targets are computed on-the-fly in the
    training loop from batch.transcriptions, not stored in Batch fields.

    Args:
        d_enc: Encoder feature dimension.
        word_vocab_size: Size of the word vocabulary (WordVocab.size).
        blank_id: CTC blank token id (default 0).
    """

    def __init__(self, d_enc: int, word_vocab_size: int, blank_id: int = 0) -> None:
        super().__init__()
        self.proj = nn.Linear(d_enc, word_vocab_size)
        self._ctc = CTCLoss(blank_id=blank_id)

    def forward(
        self,
        enc_features: torch.Tensor,        # [B, T, d_enc]
        input_lengths: torch.Tensor,       # [B]
        word_targets: torch.Tensor,        # [B, U_word]
        word_target_lengths: torch.Tensor, # [B]
    ) -> torch.Tensor:
        assert enc_features.dim() == 3, "enc_features must be [B, T, D]"
        logits = self.proj(enc_features)
        log_probs = F.log_softmax(logits.float(), dim=-1)
        return self._ctc(
            log_probs,
            targets=word_targets,
            input_lengths=input_lengths,
            target_lengths=word_target_lengths,
        )

In [ ]:
# --- Вспомогательная функция: кодирование word-targets для батча ---

def _encode_words(
    trans_list: list[str],
    wvocab: WordVocab,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Encode a list of digit-string transcriptions into word-level token tensors.

    Args:
        trans_list: List of digit strings (e.g. ["139473", "25000"]).
        wvocab: WordVocab instance.

    Returns:
        (tensor [B, U_max], lengths [B]) — padded int64 tensors.
    """
    encoded = [wvocab.encode(digits_to_words(t)) for t in trans_list]
    lengths = torch.tensor([len(e) for e in encoded], dtype=torch.int64)
    u_max = max(lengths.max().item(), 1)
    tensor = torch.zeros(len(encoded), u_max, dtype=torch.int64)
    for i, e in enumerate(encoded):
        tensor[i, : len(e)] = torch.tensor(e, dtype=torch.int64)
    return tensor, lengths

In [ ]:
# --- Manifests + vocab ---
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
vocab = CharVocab()
word_vocab = WordVocab()
print(f"Train: {len(train_records)} records. Dev: {len(dev_records)} records.")
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")

# In-domain / out-of-domain speaker split for harmonic CER metric.
in_domain_speakers = {r.spk_id for r in train_records}
out_of_domain_count = sum(1 for r in dev_records if r.spk_id not in in_domain_speakers)
in_domain_count = sum(1 for r in dev_records if r.spk_id in in_domain_speakers)
print(
    f"dev speakers: in-domain={in_domain_count} samples, "
    f"out-of-domain={out_of_domain_count} samples"
)


## Шаг 4.5. Предзагрузка аудио в RAM (опционально, сильно ускоряет обучение)

Загружает все train+dev файлы один раз, применяет resample до 16 kHz, и держит в RAM как тензоры. После этого `SpokenNumbersDataset.__getitem__` пропускает `sf.read` + `Resample` полностью.

Стоит ~3-5 минут один раз. Нужно ~2 GB RAM для GP1 (Colab: 12 GB, Kaggle: 29 GB — влезает с запасом).

In [ ]:
# FIXED — frontend / audio hyperparameters (not tunable in this experiment).
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
}
print("FIXED:", FIXED)


In [ ]:
# HP — tunable hyperparameters (single-trial values, no grid search).
HP = dict(
    lr=0.01,
    weight_decay=1e-3,
    dropout=0.1,
    d_model=256,
    subsample_factor=2,
    warmup_steps=3000,
    word_aux_weight=0.2,
    max_epochs=30,
    grad_accum=2,
    batch_size=32,
    speed_prob=0.5,
    gain_prob=0.7,
    noise_prob=0.0,
    # GPU audio augmentation
    vtlp_prob=0.3,
    vtlp_alpha_range=(0.9, 1.1),
    noise_snr_db_range=(5.0, 20.0),
    rir_prob=0.3,
    freq_mask_param=15,
    time_mask_param=25,
)
print("HP:", HP)


In [ ]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)
# Move tensors to shared memory so DataLoader worker processes reuse the
# same underlying buffers instead of copying on every fork.
for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()
print(f"AUDIO_CACHE: {len(AUDIO_CACHE)} entries")


In [ ]:
# DataLoader worker init — 1 BLAS thread per worker + per-worker RNG seed
# for the audio augmenter (avoids all workers sharing identical aug order).
import os
import random

import torch


def _worker_init(worker_id: int) -> None:
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)
    
    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


BATCH_SIZE = 32
DL_WORKERS = 4

In [ ]:
aug_cfg = AugConfig(
    speed_prob=HP["speed_prob"],
    gain_prob=HP["gain_prob"],
    seed=42,
)
train_augmenter = AudioAugmenter(aug_cfg)
gpu_aug = GPUAudioAugmenter(
    samplerate=FIXED["samplerate"],
    vtlp_prob=HP["vtlp_prob"],
    vtlp_alpha_range=HP["vtlp_alpha_range"],
    noise_prob=HP["noise_prob"],
    noise_snr_db_range=HP["noise_snr_db_range"],
    musan_root=MUSAN_ROOT,
    rir_prob=HP["rir_prob"],
    rir_root=RIR_ROOT,
).to(device)

train_ds = SpokenNumbersDataset(
    records=train_records,
    vocab=vocab,
    target_samplerate=FIXED["samplerate"],
    augmenter=train_augmenter,
    audio_cache=AUDIO_CACHE,
)
dev_ds = SpokenNumbersDataset(
    records=dev_records,
    vocab=vocab,
    target_samplerate=FIXED["samplerate"],
    augmenter=None,
    audio_cache=AUDIO_CACHE,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=DL_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=3,
    worker_init_fn=_worker_init,
)
dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=DL_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=3,
    worker_init_fn=_worker_init,
)
print(f"Train batches: {len(train_loader)}, Dev batches: {len(dev_loader)}")


## Развёрнутый тренировочный цикл — с Word-Aux CTC

**Best practice — добавлен `torch.nn.utils.clip_grad_norm_`** с `max_norm=1.0` перед каждым шагом оптимизатора.
Предотвращает взрывной рост градиентов при суммировании char-CTC и word-CTC.
Значение `max_norm=1.0` — стандартное для CTC-задач с NovoGrad.

In [ ]:
mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"], samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"], win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)
spec_aug = SpecAugmenter(freq_mask_param=HP["freq_mask_param"], num_freq_masks=2,
                          time_mask_param=HP["time_mask_param"], num_time_masks=5, seed=42)

model = QuartzNet10x4(
    vocab_size=vocab.vocab_size, d_model=HP["d_model"],
    dropout=HP["dropout"], subsample_factor=HP["subsample_factor"],
).to(device)
model = torch.compile(model, mode="default", dynamic=True)

# WordAuxCTCHead работает на финальных фичах энкодера.
# d_enc = 512 (последний блок QuartzNet10x4 возвращает log_probs из 512-канальной головки)
# Но intermediate = 256 (после B2). Здесь используем intermediate для word-aux,
# чтобы вынести word-смысл уже с середины энкодера.
word_head = WordAuxCTCHead(
    d_enc=model._d_mid,           # 256 — intermediate depth
    word_vocab_size=word_vocab.size,
    blank_id=word_vocab.BLANK_ID,
).to(device)

ctc_loss_fn = CTCLoss(blank_id=vocab.blank_id)
all_params = list(model.parameters()) + list(word_head.parameters())
optimizer = build_novograd(all_params, lr=HP["lr"], betas=(0.95, 0.5),
                            weight_decay=HP["weight_decay"])
total_steps = len(train_loader) * HP["max_epochs"]
scheduler = build_cosine_warmup(optimizer, total_steps=total_steps,
                                 warmup_steps=HP["warmup_steps"])

ckpt_dir = CKPT_ROOT / "01b_word_aux"
best_cer = float("inf")
history = []
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"WordAuxHead params: {sum(p.numel() for p in word_head.parameters()):,}")

In [ ]:
for epoch in tqdm(range(HP["max_epochs"]), desc="epochs"):
    model.train()
    word_head.train()
    spec_aug.train()
    optimizer.zero_grad()
    grad_step = 0

    for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch}", leave=False)):
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        targets = batch.targets.to(device)
        target_lengths = batch.target_lengths.to(device)

        if model.training:
            audio = gpu_aug(audio, audio_lengths)

        # Вычислить word targets на лету из batch.transcriptions
        word_targets, word_tlens = _encode_words(batch.transcriptions, word_vocab)
        word_targets = word_targets.to(device)
        word_tlens = word_tlens.to(device)

        with torch.no_grad():
            mel = mel_extractor(audio)
        mel_lengths = (
            (audio_lengths + FIXED["hop_length"] - 1) // FIXED["hop_length"]
        ).clamp(max=mel.size(-1))
        mel = spec_aug(mel, mel_lengths)

        with torch.autocast(device_type=device.type, dtype=torch.float16,
                             enabled=(device.type == "cuda")):
            out = model(mel, mel_lengths)

        with torch.autocast(device_type=device.type, enabled=False):
            loss_main = ctc_loss_fn(
                out.log_probs.float(), targets, out.output_lengths, target_lengths
            )
            # --- Word-Aux CTC: точка внедрения ---
            if out.intermediate is not None:
                loss_word = word_head(
                    out.intermediate, out.output_lengths, word_targets, word_tlens
                )
            else:
                loss_word = torch.tensor(0.0, device=device)

        loss = loss_main + HP["word_aux_weight"] * loss_word

        (loss / HP["grad_accum"]).backward()
        grad_step += 1
        if grad_step % HP["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    # Validation
    model.eval()
    word_head.eval()
    spec_aug.eval()
    all_refs, all_hyps, all_spks = [], [], []

    with torch.no_grad():
        for batch in dev_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths + FIXED["hop_length"] - 1) // FIXED["hop_length"]
            ).clamp(max=mel.size(-1))
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_refs.extend(batch.transcriptions)
            all_hyps.extend(hyps)
            all_spks.extend(batch.spk_ids)

    ref_words = [digits_to_words(r) for r in all_refs]
    val_cer = compute_cer(ref_words, all_hyps)
    in_cer, out_cer, hm_cer = compute_cer_in_out_harmonic(
        ref_words, all_hyps, all_spks, in_domain_speakers
    )

    history.append({
        "epoch": epoch,
        "val_cer": val_cer,
        "in_cer": in_cer,
        "out_cer": out_cer,
        "hm_cer": hm_cer,
    })
    tqdm.write(
        f"epoch {epoch:3d} | hm_cer={hm_cer:.4f} "
        f"(in={in_cer:.4f} out={out_cer:.4f}) | val_cer={val_cer:.4f}"
    )

    if hm_cer < best_cer:
        best_cer = hm_cer
        ckpt_path = save_best(
            model,
            meta=dict(
                arch="quartznet10x4",
                hparams=HP,
                hm_cer=hm_cer,
                val_cer=val_cer,
                in_cer=in_cer,
                out_cer=out_cer,
                epoch=epoch,
            ),
            ckpt_dir=ckpt_dir,
        )
        tqdm.write(f"  Checkpoint saved: {ckpt_path}")

if torch.cuda.is_available():
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"Peak VRAM: {peak_gb:.2f} GB")


## Итог

In [ ]:
import pandas as pd

df = pd.DataFrame(history)
print(f"Best val CER: {best_cer:.4f}")
print(f"Checkpoint dir: {ckpt_dir}")
print()
print(df.tail(10).to_string(index=False))

## Harmonic CER — DONE

Harmonic in/out digit-CER is now integrated into the training loop.
The loop tracks `hm_cer` (harmonic mean of in-domain and out-of-domain CER)
via `compute_cer_in_out_harmonic`, used for both best-model selection and `history`.
